# 🚗 CodeAlpha Data Science Internship - Task 3
## Car Price Prediction with Machine Learning (Regression Modeling)

**Author:** CodeAlpha Intern  
**Domain:** Machine Learning & Predictive Analytics  
**Task Objectives:**  
1. Ingest used car market listings with features such as original showroom price (`Present_Price`), kilometers driven, age, fuel type, transmission, and seller type.
2. Perform exploratory data analysis and feature engineering (`Car_Age`, encoding categorical features).
3. Benchmark multiple regression algorithms:
   - Linear Regression
   - Ridge & Lasso Regression
   - Random Forest Regressor
   - Gradient Boosting Regressor
4. Evaluate with $R^2$, MAE, RMSE, and 5-Fold Cross-Validation.
5. Serialize the winning model pipeline for real-time car valuation.


In [ ]:
# Import required libraries
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style="whitegrid", palette="muted")
%matplotlib inline


### 1. Data Ingestion & Feature Engineering

In [ ]:
# Load data
df = pd.read_csv(os.path.join("dataset", "car data.csv"))

# Feature engineering
df['Car_Age'] = 2024 - df['Year']

print("Dataset Shape:", df.shape)
df.head()


In [ ]:
# Summary statistics and missing values
print("Missing values:\n", df.isnull().sum())
df.describe()


### 2. Exploratory Data Analysis

In [ ]:
# Relationship: Present Price vs Selling Price
plt.figure(figsize=(8, 5))
sns.regplot(data=df, x='Present_Price', y='Selling_Price', color='#2ca02c', line_kws={'color': '#d62728'})
plt.title("Resale Price vs Showroom Present Price", fontweight='bold')
plt.xlabel("Present Price (Lakhs INR)")
plt.ylabel("Selling Price (Lakhs INR)")
plt.show()


In [ ]:
# Market Segments: Fuel Type, Transmission, Seller Type
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(x='Fuel_Type', y='Selling_Price', data=df, ax=axes[0], hue='Fuel_Type', palette='Set2', legend=False)
axes[0].set_title("Price by Fuel Type", fontweight='bold')

sns.boxplot(x='Transmission', y='Selling_Price', data=df, ax=axes[1], hue='Transmission', palette='Set2', legend=False)
axes[1].set_title("Price by Transmission", fontweight='bold')

sns.boxplot(x='Seller_Type', y='Selling_Price', data=df, ax=axes[2], hue='Seller_Type', palette='Set2', legend=False)
axes[2].set_title("Price by Seller Type", fontweight='bold')

plt.tight_layout()
plt.show()


### 3. Preprocessing Pipeline & Model Training

In [ ]:
# Feature definition
numeric_features = ['Present_Price', 'Kms_Driven', 'Car_Age', 'Owner']
categorical_features = ['Fuel_Type', 'Seller_Type', 'Transmission']

X = df[numeric_features + categorical_features]
y = df['Selling_Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

# Preprocessor
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_features)
])

# Regressors
regressors = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Lasso Regression": Lasso(alpha=0.05),
    "Random Forest": RandomForestRegressor(n_estimators=150, max_depth=6, random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(n_estimators=150, learning_rate=0.08, max_depth=4, random_state=42)
}

results = []
pipelines = {}
kf = KFold(n_splits=5, shuffle=True, random_state=42)

for name, reg in regressors.items():
    pipe = Pipeline([('preprocessor', preprocessor), ('regressor', reg)])
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring='r2')
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    results.append({
        "Model": name,
        "CV R² (Mean)": cv_r2.mean(),
        "Test R²": r2_score(y_test, y_pred),
        "MAE": mean_absolute_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred))
    })
    pipelines[name] = pipe

results_df = pd.DataFrame(results).sort_values(by="Test R²", ascending=False)
results_df


### 4. Optimal Model Diagnostics & Residual Analysis

In [ ]:
# Winning Model Evaluation
best_pipe = pipelines["Gradient Boosting"]
y_pred_best = best_pipe.predict(X_test)
residuals = y_test - y_pred_best

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
# Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, color='#1f77b4', edgecolors='k', alpha=0.8)
min_v = min(y_test.min(), y_pred_best.min())
max_v = max(y_test.max(), y_pred_best.max())
axes[0].plot([min_v, max_v], [min_v, max_v], color='#d62728', linestyle='--', linewidth=2)
axes[0].set_title("Actual vs Predicted Prices", fontweight='bold')
axes[0].set_xlabel("Actual Price (Lakhs)")
axes[0].set_ylabel("Predicted Price (Lakhs)")

# Residuals
axes[1].scatter(y_pred_best, residuals, color='#ff7f0e', edgecolors='k', alpha=0.7)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_title("Residuals vs Predicted", fontweight='bold')
axes[1].set_xlabel("Predicted Price (Lakhs)")
axes[1].set_ylabel("Residual Error")

plt.tight_layout()
plt.show()


### 5. Feature Importance Analysis

In [ ]:
# Extract feature importances
rf_pipe = pipelines["Random Forest"]
ohe = rf_pipe.named_steps['preprocessor'].named_transformers_['cat']
cat_cols = list(ohe.get_feature_names_out(categorical_features))
all_cols = numeric_features + cat_cols

imp = rf_pipe.named_steps['regressor'].feature_importances_
feat_s = pd.Series(imp, index=all_cols).sort_values(ascending=True)

plt.figure(figsize=(8, 5))
feat_s.plot(kind='barh', color='#17becf')
plt.title("Car Price Determinants: Feature Importance", fontweight='bold')
plt.xlabel("Importance Weight")
plt.show()


### 6. Pipeline Serialization & Real-Time Prediction

In [ ]:
# Save best model
joblib.dump({"pipeline": best_pipe}, os.path.join("outputs", "car_price_model.joblib"))

# Inference helper
def predict_car(present_price, kms_driven, car_age, owner, fuel_type, seller_type, transmission):
    loaded = joblib.load(os.path.join("outputs", "car_price_model.joblib"))
    df_sample = pd.DataFrame([{
        'Present_Price': present_price, 'Kms_Driven': kms_driven, 'Car_Age': car_age,
        'Owner': owner, 'Fuel_Type': fuel_type, 'Seller_Type': seller_type, 'Transmission': transmission
    }])
    return loaded["pipeline"].predict(df_sample)[0]

sample_pred = predict_car(9.85, 45000, 5, 0, 'Diesel', 'Dealer', 'Manual')
print(f"Predicted Car Resale Valuation: {sample_pred:.2f} Lakhs INR")
